<a href="https://colab.research.google.com/github/AbhipriyaMukherjee/ai-interview-assistant/blob/main/Audio_Assistant_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Required Libraries

In [ ]:
!pip install librosa soundfile numpy openai-whisper


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.9 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=49bb1a7c185fc54223ac1f75a42148540068e72e04b46278af2e7fd0fe4e8e30
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


Import Libraries

In [ ]:
import librosa
import librosa.effects
import numpy as np
import soundfile as sf
import whisper
import re
from google.colab import files


Upload Audio File

In [ ]:
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
audio_path


Saving Demo Audio_2.mp3 to Demo Audio_2.mp3


'Demo Audio_2.mp3'

Load & Preprocess Audio

In [ ]:
def load_and_preprocess_audio(path, target_sr=16000):
    audio, sr = librosa.load(path, sr=None, mono=True)

    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak

    return audio, sr


In [ ]:
audio, sr = load_and_preprocess_audio(audio_path)

print("Sample rate:", sr)
print("Audio duration (seconds):", round(len(audio) / sr, 2))
print("Min amplitude:", np.min(audio))
print("Max amplitude:", np.max(audio))


/tmp/ipython-input-3180741701.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(path, sr=None, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Sample rate: 16000
Audio duration (seconds): 34.73
Min amplitude: -0.63509965
Max amplitude: 1.0


Speech-to-Text Using Whisper

Load Whisper Model

In [ ]:
model = whisper.load_model("base")


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 154MiB/s]


Transcribe Audio

In [ ]:
result = model.transcribe(
    audio_path,
    language="en",
    fp16=False
)

transcription_text = result["text"]
print(transcription_text)


 My name is Vikramadata and I am in third year EAS. Actually I don't know what to say but I am trying my best and I hope that charity is correct and it is consistent as my vocabulary is. My exquisite vocabulary has been developed throughout my years by my poetry and exquisite writing which I would like to humbly... ...um... I am blank. Bye bye.


Basic Speech Metrics

Word Count

In [ ]:
def count_words(text):
    return len(text.strip().split())

total_words = count_words(transcription_text)
print("Total words:", total_words)


Total words: 66


Speaking Rate (WPM)

In [ ]:
def speaking_rate_wpm(word_count, duration_seconds):
    minutes = duration_seconds / 60
    return word_count / minutes if minutes > 0 else 0

audio_duration = len(audio) / sr
wpm = speaking_rate_wpm(total_words, audio_duration)

print("Speaking rate (WPM):", round(wpm, 1))


Speaking rate (WPM): 114.0


Pause Detection

In [ ]:
def detect_pauses_with_timestamps(
    audio,
    sr,
    silence_db=25,
    short_pause=(0.3, 0.7),
    long_pause_threshold=0.7
):
    intervals = librosa.effects.split(audio, top_db=silence_db)

    pauses = []
    prev_end = 0

    for start, end in intervals:
        pause_duration = (start - prev_end) / sr
        pause_start_time = prev_end / sr

        if pause_duration > 0:
            pauses.append({
                "duration": pause_duration,
                "start_time": pause_start_time
            })

        prev_end = end

    short_pauses = [
        p for p in pauses
        if short_pause[0] <= p["duration"] < short_pause[1]
    ]

    long_pauses = [
        p for p in pauses
        if p["duration"] >= long_pause_threshold
    ]

    total_pause_time = sum(p["duration"] for p in pauses)

    return short_pauses, long_pauses, total_pause_time


In [ ]:
short_pauses, long_pauses, total_pause_time = detect_pauses_with_timestamps(audio, sr)

pause_ratio = total_pause_time / audio_duration

print("Short pauses:", len(short_pauses))
print("Long pauses:", len(long_pauses))
print("Total pause time:", round(total_pause_time, 2))
print("Pause ratio:", round(pause_ratio, 3))

print("\nLong pause timestamps:")
for p in long_pauses:
    print(f"- {round(p['start_time'],2)}s → {round(p['duration'],2)}s")


Short pauses: 4
Long pauses: 0
Total pause time: 4.16
Pause ratio: 0.12

Long pause timestamps:


Filler Word Detection

In [ ]:
FILLER_WORDS = [
    "uh", "um", "erm",
    "like", "you know",
    "actually", "basically",
    "i mean"
]

def detect_fillers(text, filler_list):
    text = text.lower()
    filler_counts = {}

    for filler in filler_list:
        pattern = r"\b" + re.escape(filler) + r"\b"
        filler_counts[filler] = len(re.findall(pattern, text))

    total_fillers = sum(filler_counts.values())
    return filler_counts, total_fillers


In [ ]:
filler_counts, total_fillers = detect_fillers(
    transcription_text,
    FILLER_WORDS
)

fillers_per_100_words = (
    (total_fillers / total_words) * 100
    if total_words > 0 else 0
)

print("Filler breakdown:", filler_counts)
print("Total fillers:", total_fillers)
print("Fillers per 100 words:", round(fillers_per_100_words, 2))


Filler breakdown: {'uh': 0, 'um': 1, 'erm': 0, 'like': 1, 'you know': 0, 'actually': 1, 'basically': 0, 'i mean': 0}
Total fillers: 3
Fillers per 100 words: 4.55


Consistency & Clarity Metrics

Speaking Consistency (Segment Length Variance)

In [ ]:
segment_lengths = [
    (seg["end"] - seg["start"])
    for seg in result["segments"]
]

consistency_std = np.std(segment_lengths)
print("Speaking consistency (std):", round(consistency_std, 2))


Speaking consistency (std): 4.4


Clarity Score (Rule-Based Proxy)

In [ ]:
def clarity_score(pause_ratio, fillers_per_100, wpm):
    score = 100

    if pause_ratio > 0.35:
        score -= 30
    elif pause_ratio > 0.25:
        score -= 15

    if fillers_per_100 > 8:
        score -= 30
    elif fillers_per_100 > 4:
        score -= 15

    if wpm < 90 or wpm > 180:
        score -= 15

    return max(score, 0)


Weighted Scoring & Final Aggregation

In [ ]:
WEIGHTS = {
    "fluency": 0.35,
    "clarity": 0.30,
    "consistency": 0.20,
    "delivery": 0.15
}


Individual Scores

In [ ]:
fluency_score = max(
    100
    - (pause_ratio * 100)
    - (fillers_per_100_words * 5)
    - (len(long_pauses) * 5),
    0
)

clarity_numeric = clarity_score(
    pause_ratio,
    fillers_per_100_words,
    wpm
)

consistency_score = max(
    100 - (consistency_std * 10),
    0
)

delivery_score = 100 if 90 <= wpm <= 180 else 70


Final Speech Score

In [ ]:
final_score = (
    fluency_score * WEIGHTS["fluency"]
    + clarity_numeric * WEIGHTS["clarity"]
    + consistency_score * WEIGHTS["consistency"]
    + delivery_score * WEIGHTS["delivery"]
)

print("\n--- FINAL SPEECH EVALUATION ---")
print("Fluency score:", round(fluency_score, 1))
print("Clarity score:", round(clarity_numeric, 1))
print("Consistency score:", round(consistency_score, 1))
print("Delivery score:", round(delivery_score, 1))
print("\nFinal Speech Score:", round(final_score, 1), "/ 100")



--- FINAL SPEECH EVALUATION ---
Fluency score: 65.3
Clarity score: 85
Consistency score: 56.0
Delivery score: 100

Final Speech Score: 74.6 / 100


Feedback Generation

In [ ]:
def generate_feedback(
    pause_ratio,
    fillers_per_100,
    long_pause_count,
    wpm
):
    feedback = []

    if pause_ratio > 0.25:
        feedback.append(
            "Frequent pauses were observed, which may indicate hesitation."
        )

    if fillers_per_100 > 5:
        feedback.append(
            "Frequent filler words reduced overall speech fluency."
        )

    if long_pause_count > 2:
        feedback.append(
            "Long pauses during responses suggest a need for smoother flow."
        )

    if wpm < 90:
        feedback.append(
            "Speaking rate was slow; increasing pace may improve confidence."
        )

    if wpm > 180:
        feedback.append(
            "Speaking rate was very fast; slowing down may improve clarity."
        )

    if not feedback:
        feedback.append(
            "Speech delivery was clear and fluent overall."
        )

    return feedback


Score Interpretation

In [ ]:
def interpret_score(score):
    if score >= 85:
        return "Excellent"
    elif score >= 70:
        return "Good"
    elif score >= 50:
        return "Needs Improvement"
    else:
        return "Poor"


Final Report

In [ ]:
feedback = generate_feedback(
    pause_ratio,
    fillers_per_100_words,
    len(long_pauses),
    wpm
)

score_label = interpret_score(final_score)

print("\n================ FINAL SPEECH REPORT ================")
print(f"Final Speech Score : {round(final_score,1)} / 100")
print(f"Performance Level  : {score_label}")
print("\nKey Feedback:")

for f in feedback:
    print("-", f)



================ FINAL SPEECH REPORT ================
Final Speech Score : 74.6 / 100
Performance Level  : Good

Key Feedback:
- Speech delivery was clear and fluent overall.
